**COMPACT MODEL FOR THE APP**

In [1]:
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("train.csv")

numeric = ["GrLivArea", "OverallQual", "GarageCars", "TotalBsmtSF", "YearBuilt", "FullBath", "BedroomAbvGr"]
categorical = ["Neighborhood"]
features = numeric + categorical

print("Missing values in the chosen columns:", df[features].isna().sum().sum())

train, test = train_test_split(df, test_size=0.2, random_state=42)

prep = ColumnTransformer(
    [("hood", OneHotEncoder(handle_unknown="ignore"), categorical)],
    remainder="passthrough"
)
model = Pipeline([
    ("prep", prep),
    ("gb", GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)),
])

model.fit(train[features], train["SalePrice"])
pred = model.predict(test[features])

print("Compact model | MAE:", round(mean_absolute_error(test["SalePrice"], pred)),
      " | R2:", round(r2_score(test["SalePrice"], pred), 2))
print("Full model for comparison | MAE: 16626 | R2: 0.9")
print("Scikit-learn version:", sklearn.__version__)

Missing values in the chosen columns: 0
Compact model | MAE: 17635  | R2: 0.91
Full model for comparison | MAE: 16626 | R2: 0.9
Scikit-learn version: 1.6.1


**SAVE THE MODEL AND TEST LOADING IT**

In [5]:
import os, json
import joblib
import numpy as np

joblib.dump(model, "house_price_model.joblib")

meta = {
    "features": features,
    "neighborhoods": sorted(df["Neighborhood"].unique().tolist()),
    "ranges": {c: [int(df[c].min()), int(df[c].max())] for c in numeric},
    "test_mae": round(mean_absolute_error(test["SalePrice"], pred)),
    "sklearn_version": sklearn.__version__,
    "pandas_version": pd.__version__,
}
with open("model_info.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Model file size (MB):", round(os.path.getsize("house_price_model.joblib") / 1e6, 2))
print("Neighborhoods:", len(meta["neighborhoods"]))
print("Ranges:", meta["ranges"])

# Test: load the file back and predict one house
loaded = joblib.load("house_price_model.joblib")
example = pd.DataFrame([{
    "GrLivArea": 1800, "OverallQual": 7, "GarageCars": 2, "TotalBsmtSF": 900,
    "YearBuilt": 2000, "FullBath": 2, "BedroomAbvGr": 3, "Neighborhood": "CollgCr"}])
print("Prediction for the example house:", round(loaded.predict(example)[0]))
print("Loaded model matches the original:",
      np.allclose(loaded.predict(test[features]), model.predict(test[features])))

Model file size (MB): 0.39
Neighborhoods: 25
Ranges: {'GrLivArea': [334, 5642], 'OverallQual': [1, 10], 'GarageCars': [0, 4], 'TotalBsmtSF': [0, 6110], 'YearBuilt': [1872, 2010], 'FullBath': [0, 3], 'BedroomAbvGr': [0, 8]}
Prediction for the example house: 202992
Loaded model matches the original: True


**THE APP PLAN**

## The app plan

**Inputs (the form):**
- Living area in sq ft (334 to 5,642)
- Overall quality, 1 to 10
- Garage size, 0 to 4 cars
- Basement area in sq ft (0 to 6,110)
- Year built (1872 to 2010)
- Full bathrooms (0 to 3)
- Bedrooms (0 to 8)
- Neighborhood (dropdown of 25)

**Output:**
- A predicted price
- A range around it, based on the model's typical error of about 17,600

**Note shown on the page:**
This model was trained on house sales in Ames, Iowa. It is a learning project, not valuation advice.